# Les 18: AI-agenten beveiligen met cryptografische ontvangstbewijzen

## Praktijknotebook

Deze notebook doorloopt vier oefeningen:

1. **Onderteken je eerste ontvangstbewijs** voor een agent tool-aanroep en verifieer het.
2. **Manipuleer het ontvangstbewijs** en zie de verificatie mislukken.
3. **Bouw een keten van drie ontvangstbewijzen** en bevestig de integriteit van de keten.
4. **Omsluit een Microsoft Agent Framework tool-aanroep** zodat elke actie een ontvangstbewijs genereert.

Alle cryptografische primitieve functies zijn geïmporteerd uit goed onderhouden bibliotheken (`pynacl` voor Ed25519, `jcs` voor RFC 8785 canonieke JSON, `hashlib` uit de Python standaardbibliotheek voor SHA-256). De logica van het ontvangstbewijs zelf is eenvoudige Python die je kunt lezen en aanpassen.

Voer de cellen op volgorde uit. Elke sectie is kort en op zichzelf staand.


## Installatie

Installeer de twee afhankelijkheden. Beide hebben permissieve licenties (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## Hulpprogramma's

Deze twee hulpprogramma's behandelen base64url-codering (zonder opvulling) en SHA-256-hashing van willekeurige objecten. Ze houden de rest van het notitieboekje gericht op de bonlogica zelf.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Sectie 1: Onderteken je eerste ontvangstbewijs

Stel je voor dat onze agent voor **Contoso Travel** net vluchten van Sydney naar Los Angeles heeft opgezocht voor een klant. We willen de verklaring van de gateway over dit tool-aanroep vastleggen zodat een toekomstige auditor deze kan verifiëren onder een apart vastgepinde uitgeverssleutel.

### Stap 1.1: Genereer een ondertekeningssleutel

In productie zou de ondertekeningssleutel van de agent in een hardware security module (HSM), Azure Key Vault of een vergelijkbare beveiligde opslag worden bewaard. Voor deze les genereren we een nieuwe sleutel in het geheugen.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### Stap 1.2: Bouw de ontvangstpayload

De payload bevat alles wat we willen dat de ontvangstbewijs bevestigt: wie handelde, welk hulpmiddel, met welke argumenten, wat er terugkwam, onder welk beleid, en wanneer. We hashen de argumenten en het resultaat in plaats van ze inline op te nemen zodat de ontvangstbewijs geen gevoelige inhoud lekt.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### Stap 1.3: Onderteken en stel de ontvangstbevestiging samen

Drie stappen:

1. Canonicaliseer de payload met JCS zodat twee implementaties die dezelfde logische ontvangstbevestiging produceren, identieke bytes genereren.
2. Onderteken de canonieke bytes rechtstreeks met de Ed25519 privésleutel. PureEdDSA heeft intern een hash van het bericht, dus een extra pre-hash zou het protocol veranderen.

De handtekening wordt vervolgens aan de originele payload toegevoegd om de definitieve ontvangstbevestiging te maken.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### Stap 1.4: Verifieer het ontvangstbewijs

Verificatie keert het proces om. We verwijderen de handtekening, berekenen de canonieke bytes opnieuw en controleren de handtekening tegen de openbare sleutel van de uitgever.

Verificatie is offline: er wordt geen service aangeroepen en er is geen netwerktoegang. Het heeft nog wel één ding nodig naast het ontvangstbewijs: de openbare sleutel van de uitgever, verkregen via een kanaal dat de auditor al vertrouwt, zoals een JWK Set die de uitgever publiceert of een waarde die vooraf vastgezet is. Het `public_key`-veld in het ontvangstbewijs kan die rol niet vervullen, omdat degene die het ontvangstbewijs schrijft ook dat veld schrijft. Sectie 2 toont de vervalsing die dit voorkomt.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


Je zou `Receipt is valid: True`, `Pre-hashed receipt valid: False` en `Malformed receipts rejected: True` moeten zien. De positieve test bewijst dat het directe JCS-handtekeningpad werkt; de negatieve controle zorgt ervoor dat de handtekening-bereikregel uitvoerbaar wordt in plaats van als tekst te blijven staan; de laatste regel toont dat foutieve invoer `False` teruggeeft in plaats van een uitzondering te gooien.


## Sectie 2: Knutselen aan de bon

Het hele punt van bonnen is dat ze aantonen wanneer ze zijn gemanipuleerd. Laten we dat bewijzen.

We zullen precies één teken van de bon aanpassen en zien dat de verificatie faalt.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### Wat is er net gebeurd?

Toen we `policy_id` wijzigden, veranderden de canonieke bytes. De handtekening (die over de oorspronkelijke canonieke bytes was) komt niet langer overeen. Verificatie geeft correct `False` terug.

Er is geen manier om een ondertekend veld te wijzigen, wat betekent alles buiten het `signature` object, en toch de ontvangstbewijs te laten verifiëren tegen de publieke sleutel van de uitgever, tenzij de aanvaller de private sleutel van de uitgever heeft. Het `signature` object zelf is niet ondertekend: `sig` is de handtekening, en `verify_receipt` negeert `public_key`, dus alleen dat veld aanpassen verandert niets. Zolang de private sleutel zich in een sleutelkluis bevindt en verificateurs de onafhankelijk gepubliceerde sleutel van de uitgever gebruiken, is het onmogelijk om een wijziging aan een ondertekend veld te verbergen. Die voorwaarde is belangrijk: de volgende cel toont een aangepast ontvangstbewijs dat perfect verifieert tegen een andere sleutel.

Probeer het zelf: wijzig het `tool_name` of `agent_id` of `timestamp` in de cel hierboven en voer opnieuw uit. Iedere wijziging aan een ondertekend veld levert een ongeldig ontvangstbewijs op.


### De manipulatie die de handtekening niet vangt

De aanvaller hierboven heeft de handtekening niet opnieuw gezet. Een zorgvuldiger iemand zou: het ontvangstbewijs wijzigen,
het ondertekenen met een zelfgegenereerde sleutel, en die sleutel invullen in het
`public_key` veld. Het resultaat is een ontvangstbewijs waarvan de handtekening volledig authentiek is
voor de inhoud ervan.

Voer dit uit en kijk welke van de twee controles het detecteert.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


Je zou `True` moeten zien en daarna `False`. De drie `assert` regels maken dat een
controle: de cel geeft een foutmelding als de vervalsing ooit controleert tegen de
sleutel van de uitgever.

De vervalsing is intern consistent. De handtekening komt echt overeen met de
inhoud en de sleutel die het draagt, dus een verifier die de sleutel uit de
bon leest, rapporteert deze als geldig. Er lijkt niets mis met de bon,
want er is niets inconsistent aan.

Het enige dat het detecteert is het controleren tegen een sleutel waarop de aanvaller
geen invloed heeft gehad. Daarom neemt `verify_receipt` de sleutel als argument: een
functie met de handtekening `verify_receipt(receipt)` kan niet veilig geschreven worden,
omdat de enige sleutel waartoe het toegang zou hebben de door een aanvaller verstrekte is.

Dezelfde bon wordt opgeslagen als `sample_receipts/04_resigned_receipt.json` als je
die buiten de notebook wilt inspecteren.


## Sectie 3: Ontvangsten aan elkaar koppelen

Een enkele ontvangst beschermt één actie. De meeste agenten voeren veel acties uit. Om de hele reeks weerbestendig te maken, koppelen we elke ontvangst aan de vorige door de hash van de vorige ontvangst op te nemen in de payload van de nieuwe ontvangst.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Als iemand een ontvangst binnen de gepresenteerde reeks verwijdert of herordent, wordt de volgende schakel verbroken. Het detecteren van afkapping of door ondertekenaar gecontroleerde achtervoegselherschrijving vereist ook een vastgezette huidige kop, een externe controlepost, of een geauthenticeerde populatieverbintenis.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Breek nu de keten door te knoeien met de middelste ontvangstbevestiging en verifieer opnieuw. De gemanipuleerde ontvangstbevestiging faalt zijn handtekeningcontrole, EN de volgende ontvangstbevestiging faalt zijn kettingkoppelingcontrole (omdat zijn `previous_receipt_hash` niet langer overeenkomt met de hash van de gewijzigde middelste ontvangstbevestiging).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


Ontvangstbewijs 0 verifieert nog steeds (het is niet gewijzigd en heeft geen voorganger waarop het moet vertrouwen). Ontvangstbewijs 1 faalt zijn handtekeningcontrole omdat we `tool_args_hash` hebben gewijzigd. Ontvangstbewijs 2 faalt zijn ketenlinkcontrole omdat zijn `previous_receipt_hash` is berekend op basis van het oorspronkelijke (nu gewijzigde) ontvangstbewijs 1.

Zelfs als een aanvaller het gewijzigde ontvangstbewijs 1 opnieuw ondertekent, faalt de nieuwe handtekening bij controle met de sleutel van de uitgever, tenzij ze de privésleutel van de uitgever bezitten (een nieuwe ondertekening met hun eigen sleutel faalt, zoals Sectie 2 aantoonde), en de ketenlink-mismatch in ontvangstbewijs 2 zou de manipulatie nog steeds blootleggen. Om de wijziging te verbergen, zou de aanvaller elk ontvangstbewijs vanaf het wijzigingspunt opnieuw moeten ondertekenen, wat vereist dat ze de privésleutel van de uitgever bezitten.


## Sectie 4: Wikkel een agent tool-aanroep in met ontvangstbevestiging ondertekening

In een echte implementatie wil je niet dat elke agent auteur onthoudt om `make_receipt` aan te roepen. Je wilt dat ontvangstbevestiging automatisch gebeurt bij elke tool-aanroep.

Hier is het eenvoudigste patroon: een wrapper-klasse die elke oproepbare toolfunctie pakt en een versie ervan teruggeeft die een ontvangst aflevert. Dit past zich aan elk agentframework aan, inclusief het Microsoft Agent Framework (`agent_framework.foundry`).

Als je geen Microsoft Foundry-project hebt opgezet, demonstreert de lokale mock hieronder nog steeds het patroon.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Integreren met het Microsoft Agent Framework

De `ReceiptedTool` wrapper hierboven is framework-agnostisch. Om deze te gebruiken binnen een agent die is gebouwd met het Microsoft Agent Framework, registreert u de ingepakte functie als een tool. Een schets (u zou de mock vervangen door een echte registratie van een Microsoft Foundry-tool):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

Het agent framework hoeft niets te weten over ontvangstbewijzen. Het ondertekenen van ontvangstbewijzen is om de tool heen gewikkeld, niet geïntegreerd in het framework. Zo voegt u herkomst toe aan bestaande agentcode zonder de agent te herschrijven.


## Samenvatting en stretch-uitdaging

Je hebt:

- Een Ed25519-sleutelpaar gegenereerd.
- Een ontvangstbewijs voor een agenttool-aanroep opgebouwd en ondertekend.
- Het ontvangstbewijs offline geverifieerd met alleen de openbare sleutel van de uitgever, onafhankelijk verkregen van het ontvangstbewijs.
- Een ontvangstbewijs gesaboteerd en geconstateerd dat de verificatie faalt.
- Een hash-gekoppelde reeks van drie ontvangstbewijzen opgebouwd.
- Het midden van de keten gesaboteerd en gezien dat zowel de handtekening als de ketting-link faalden.
- Een toolfunctie ingepakt met automatische ondertekening van ontvangstbewijzen.

**Stretch-uitdaging.** Breid het ontvangstbewijs-schema uit met een `request_id`-veld (een UUID voor gedistribueerde tracing). Werk `make_receipt` bij om dit op te nemen en bevestig dat ontvangstbewijzen nog steeds end-to-end verifiëren. Wijzig daarna het veld na ondertekening en bevestig dat verificatie faalt. Dit dwingt je om te begrijpen hoe elke byte van de canonieke codering bijdraagt aan de handtekening.

**Belangrijke grens.** Onder een vastgezette sleutel en ketencontext bewijzen ontvangstbewijzen handtekeningtoeschrijving, inhoudsintegriteit, en een link naar de gepresenteerde voorganger. Ze bewijzen NIET uitvoering, bronwaarheid, vertrouwde tijd, compleetheid, beleidsbeoordeling of correctheid. Ontvangstbewijzen zijn een fundament. Governance is het systeem dat je daarop bouwt.

Lees de les-README opnieuw met die grens in gedachten. De meest voorkomende vergissing die teams maken met ontvangstbewijzen is aannemen dat "we hebben ontvangstbewijzen" betekent "we worden bestuurd". Dat is niet zo. Ontvangstbewijzen maken agentgedrag controleerbaar. Ze maken het niet correct.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
